## imports & constants

In [1]:
import pandas as pd
import re
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
import seaborn as sns

DATA_PATH = './dataset.csv'
LABELS_MAP = {
    0: 'Politics',
    1: 'Sport',
    2: 'Technology',
    3: 'Entertainment',
    4: 'Business'
}

In [2]:
words_df = pd.read_csv('words.csv', header=None)    
vocabulary_list = words_df[0].astype(str).str.lower().tolist()

## q7

In [3]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def q7_load_and_preprocess():
    df = pd.read_csv(DATA_PATH)
    print("Q7 | Dataset shape:", df.shape)
    df['clean_text'] = df['Text'].apply(preprocess)
    return df

In [4]:
df = q7_load_and_preprocess()
df

Q7 | Dataset shape: (2225, 2)


,Text,Label,clean_text
0,Budget to set scene for election\n \n Gordon B...,0,budget to set scene for election gordon brown ...
1,Army chiefs in regiments decision\n \n Militar...,0,army chiefs in regiments decision military chi...
2,Howard denies split over ID cards\n \n Michael...,0,howard denies split over id cards michael howa...
3,Observers to monitor UK election\n \n Minister...,0,observers to monitor uk election ministers wil...
4,Kilroy names election seat target\n \n Ex-chat...,0,kilroy names election seat target ex chat show...
...,...,...,...
2220,India opens skies to competition\n \n India wi...,4,india opens skies to competition india will al...
2221,Yukos bankruptcy 'not US matter'\n \n Russian ...,4,yukos bankruptcy not us matter russian authori...
2222,Survey confirms property slowdown\n \n Governm...,4,survey confirms property slowdown government f...
2223,High fuel prices hit BA's profits\n \n British...,4,high fuel prices hit ba s profits british airw...


## q8

In [5]:
def q8_top30_words(df, fname, stopwords=False):
    if stopwords:
        stopwords = ENGLISH_STOP_WORDS
    else:
        stopwords = {}
        
    stop_words_set = set(stopwords)

    all_words = []
    for text in df['clean_text']:
        words = [w for w in text.split() if w not in stop_words_set and len(w) > 1]
        all_words.extend(words)

    counter = Counter(all_words)
    top30 = counter.most_common(30)
    print("\nQ8 | Top 30 frequent words:", top30)

    words, counts = zip(*top30)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(words, counts, color='steelblue')
    plt.xticks(rotation=60, ha='right')
    ax.set_title('Top 30 Most Frequent Words (all documents)')
    ax.set_ylabel('Frequency')
    plt.tight_layout()
    # plt.savefig(f'media/{fname}.png', dpi=130)
    plt.close(fig)
    
    return top30

In [6]:
q8_top30_words(df, "q8_top30_wo_stopwords")


Q8 | Top 30 frequent words: [('the', 52636), ('to', 25113), ('of', 20008), ('and', 18612), ('in', 17734), ('for', 8945), ('is', 8555), ('that', 8257), ('it', 7893), ('on', 7625), ('said', 7255), ('was', 6028), ('he', 5939), ('be', 5805), ('with', 5354), ('as', 4981), ('has', 4957), ('have', 4772), ('at', 4638), ('by', 4516), ('will', 4473), ('but', 4421), ('are', 4401), ('from', 3535), ('not', 3484), ('they', 3085), ('his', 3026), ('we', 3006), ('mr', 3005), ('this', 2847)]


[('the', 52636),
 ('to', 25113),
 ('of', 20008),
 ('and', 18612),
 ('in', 17734),
 ('for', 8945),
 ('is', 8555),
 ('that', 8257),
 ('it', 7893),
 ('on', 7625),
 ('said', 7255),
 ('was', 6028),
 ('he', 5939),
 ('be', 5805),
 ('with', 5354),
 ('as', 4981),
 ('has', 4957),
 ('have', 4772),
 ('at', 4638),
 ('by', 4516),
 ('will', 4473),
 ('but', 4421),
 ('are', 4401),
 ('from', 3535),
 ('not', 3484),
 ('they', 3085),
 ('his', 3026),
 ('we', 3006),
 ('mr', 3005),
 ('this', 2847)]

In [7]:
q8_top30_words(df, "q8_top30_w_stopwords", True)


Q8 | Top 30 frequent words: [('said', 7255), ('mr', 3005), ('year', 2310), ('people', 2045), ('new', 1978), ('time', 1322), ('world', 1201), ('government', 1160), ('uk', 1115), ('years', 1003), ('best', 974), ('bn', 958), ('just', 957), ('make', 945), ('told', 911), ('film', 890), ('like', 879), ('game', 871), ('music', 839), ('labour', 804), ('bbc', 778), ('set', 762), ('number', 760), ('way', 740), ('added', 733), ('market', 702), ('says', 687), ('company', 686), ('home', 663), ('election', 662)]


[('said', 7255),
 ('mr', 3005),
 ('year', 2310),
 ('people', 2045),
 ('new', 1978),
 ('time', 1322),
 ('world', 1201),
 ('government', 1160),
 ('uk', 1115),
 ('years', 1003),
 ('best', 974),
 ('bn', 958),
 ('just', 957),
 ('make', 945),
 ('told', 911),
 ('film', 890),
 ('like', 879),
 ('game', 871),
 ('music', 839),
 ('labour', 804),
 ('bbc', 778),
 ('set', 762),
 ('number', 760),
 ('way', 740),
 ('added', 733),
 ('market', 702),
 ('says', 687),
 ('company', 686),
 ('home', 663),
 ('election', 662)]

## q9

In [8]:
def q9_wordcloud(df, fname):
    extra_stop = ['said', 'mr', 'told', 'year', 'years', 'new', 'time', 'best', 'world']
    stop_words = list(ENGLISH_STOP_WORDS.union(extra_stop))

    vectorizer = TfidfVectorizer(stop_words=stop_words, max_df=0.9, min_df=2)
    tfidf = vectorizer.fit_transform(df['clean_text'])
    scores = np.asarray(tfidf.sum(axis=0)).ravel()
    vocab = vectorizer.get_feature_names_out()

    top_idx = np.argsort(scores)[::-1][:20]
    top_words = [(vocab[i], scores[i]) for i in top_idx]

    random.seed(7)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')
    
    max_score = max(s for _, s in top_words)
    min_score = min(s for _, s in top_words)
    placed = []

    def overlaps(x, y, w, h, placed):
        for (px, py, pw, ph) in placed:
            if abs(x - px) < (w + pw) / 2 * 0.75 and abs(y - py) < (h + ph) / 2 * 0.75:
                return True
        return False

    for word, score in top_words:
        size = 16 + 55 * (score - min_score) / (max_score - min_score)
        
        for _ in range(300):
            x = random.uniform(0.1, 0.9)
            y = random.uniform(0.1, 0.9)
            w = len(word) * size * 0.011
            h = size * 0.02
            if not overlaps(x, y, w, h, placed):
                placed.append((x, y, w, h))
                break
        
        color = plt.cm.tab20(random.random())
        ax.text(x, y, word, fontsize=size, ha='center', va='center', color=color,
                fontweight='bold', fontfamily='sans-serif', transform=ax.transAxes)

    ax.set_title('Top 20 Words Wordcloud', fontsize=16)
    plt.tight_layout()
    # plt.savefig(f'media/{fname}.png', dpi=130)
    plt.close(fig)
    
    return top_words

In [9]:
q9_wordcloud(df, "q9_wordcloud")

[('people', np.float64(38.11888684148821)),
 ('film', np.float64(32.611031682135625)),
 ('bn', np.float64(30.644135278700738)),
 ('government', np.float64(29.59906836926095)),
 ('uk', np.float64(27.94779443327599)),
 ('labour', np.float64(27.13517911679139)),
 ('game', np.float64(25.897526690640593)),
 ('election', np.float64(23.87570499905268)),
 ('music', np.float64(23.478663030251198)),
 ('blair', np.float64(22.539061919384995)),
 ('england', np.float64(21.92580846308045)),
 ('number', np.float64(21.348826821797545)),
 ('just', np.float64(21.1935742116639)),
 ('make', np.float64(21.02083024538935)),
 ('party', np.float64(20.719867288739334)),
 ('company', np.float64(20.37872593377917)),
 ('market', np.float64(20.310467710648393)),
 ('bbc', np.float64(20.147181031725232)),
 ('like', np.float64(19.879363587634728)),
 ('set', np.float64(19.85851204835371))]

## q10

In [10]:
def q10_bow_matrix(df, vocabulary_list):
    vectorizer = CountVectorizer(vocabulary=vocabulary_list)
    bow = vectorizer.fit_transform(df['clean_text'])
    bow_matrix = bow.toarray()
    
    print(f"\nQ10 | BoW matrix shape: {bow_matrix.shape}  (docs x |vocab|={len(vocabulary_list)})")
    
    return bow_matrix

In [11]:
bow_full = q10_bow_matrix(df, vocabulary_list)
bow_full


Q10 | BoW matrix shape: (2225, 53)  (docs x |vocab|=53)


array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2225, 53))

## Dataset Partitioning

In [12]:
def split_2000_225(df, bow_matrix):    
    """Split literally: first 2000 rows = working set, last 225 = held-out test."""
    df_train = df.iloc[:2000].reset_index(drop=True)
    df_test = df.iloc[2000:].reset_index(drop=True)
    bow_train = bow_matrix[:2000]
    bow_test = bow_matrix[2000:]
    print("Split | train:", df_train.shape, bow_train.shape,
          "| test:", df_test.shape, bow_test.shape)
    return df_train, df_test, bow_train, bow_test

def split_2000_225_shuffle(df, bow_matrix):
    df_train, df_test, bow_train, bow_test = train_test_split(
        df, 
        bow_matrix, 
        train_size=2000, 
        shuffle=True, 
        random_state=42
    )
    
    df_train = df_train.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    print("Split | train:", df_train.shape, bow_train.shape,
          "| test:", df_test.shape, bow_test.shape)
          
    return df_train, df_test, bow_train, bow_test

In [13]:
df_train, df_test, bow_train, bow_test = split_2000_225_shuffle(df, bow_full)

Split | train: (2000, 3) (2000, 53) | test: (225, 3) (225, 53)


## q11

In [14]:
def q11_standardize_and_svd(bow_matrix):
    scaler = StandardScaler()
    bow_scaled = scaler.fit_transform(bow_matrix)
    
    U, S, Vt = np.linalg.svd(bow_scaled, full_matrices=False)
    
    print("=== SVD Matrices Dimensions ===")
    print(f"Original Scaled Matrix (M x N) : {bow_scaled.shape}")
    print(f"U Matrix (Left Singular)       : {U.shape}")
    print(f"S Array (Singular Values)      : {S.shape}")
    print(f"V^T Matrix (Right Singular)    : {Vt.shape}")
    
    return bow_scaled, U, S, Vt, scaler

In [15]:
bow_scaled, U, S, Vt, scaler = q11_standardize_and_svd(bow_train)

=== SVD Matrices Dimensions ===
Original Scaled Matrix (M x N) : (2000, 53)
U Matrix (Left Singular)       : (2000, 53)
S Array (Singular Values)      : (53,)
V^T Matrix (Right Singular)    : (53, 53)


## q12

In [16]:
def q12_truncated_svd(bow_scaled, U, S, Vt, fname, fname2,variance_threshold=0.90):
    total_variance = np.sum(S ** 2)
    cumulative_variance = np.cumsum(S ** 2) / total_variance
    
    k_optimal = np.argmax(cumulative_variance >= variance_threshold) + 1
    
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(cumulative_variance, color='steelblue', linewidth=2)
    ax.axhline(y=variance_threshold, color='red', linestyle='--', label=f'Threshold ({variance_threshold*100}%)')
    ax.axvline(x=k_optimal, color='green', linestyle='--', label=f'Optimal k = {k_optimal}')
    
    ax.set_title('Cumulative Explained Variance by Singular Values', fontsize=14)
    ax.set_xlabel('Number of Singular Values (k)', fontsize=12)
    ax.set_ylabel('Cumulative Variance', fontsize=12)
    ax.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'media/{fname}', dpi=130)
    plt.close(fig)    

    print(f"Suggested k to retain {variance_threshold*100}% variance: {k_optimal}")
    
    U_truncated = U[:, :k_optimal]
    S_truncated = np.diag(S[:k_optimal])
    Vt_truncated = Vt[:k_optimal, :]
    
    print("\n=== Truncated Matrices Dimensions ===")
    print(f"U_k shape  : {U_truncated.shape}")
    print(f"S_k shape  : {S_truncated.shape}")
    print(f"Vt_k shape : {Vt_truncated.shape}")
    
    bow_reconstructed = np.dot(U_truncated, np.dot(S_truncated, Vt_truncated))
    
    reconstruction_error = np.linalg.norm(bow_scaled - bow_reconstructed, 'fro')
    print(f"Reconstruction Error : {reconstruction_error:.4f}")
    
    return U_truncated, S_truncated, Vt_truncated, bow_reconstructed, reconstruction_error

In [17]:
start_time = time.time()
U_truncated, S_truncated, Vt_truncated, bow_reconstructed, reconstruction_error = q12_truncated_svd(bow_scaled, U, S, Vt, "q12_svd_threshold", "q12_svd_scatter")
trunc_time = time.time() - start_time

Suggested k to retain 90.0% variance: 42

=== Truncated Matrices Dimensions ===
U_k shape  : (2000, 42)
S_k shape  : (42, 42)
Vt_k shape : (42, 53)
Reconstruction Error : 98.5440


## q13

In [18]:
def q13_randomized_svd(A, k, p=5, random_state=42):
    np.random.seed(random_state)
        
    m, n = A.shape
    r = k + p
        
    Omega = np.random.normal(size=(n, r))
    
    Y = A @ Omega
    Q, _ = np.linalg.qr(Y)    
    B = Q.T @ A
    
    U_tilde, S, Vt = np.linalg.svd(B, full_matrices=False)
    
    U = Q @ U_tilde
    
    U_k = U[:, :k]
    S_k = S[:k]
    Vt_k = Vt[:k, :]
    
    return U_k, S_k, Vt_k

In [19]:
U_r, S_r, Vt_r = q13_randomized_svd(bow_scaled, 42)

## q14

In [20]:
def q14_compare_svd(X_scaled, k_optimal, error_truncated, time_truncated):
    print(f"=== SVD Comparison (k = {k_optimal}) ===")
    
    start_time = time.time()
    U_r, S_r, Vt_r = q13_randomized_svd(X_scaled, k=k_optimal)
    rand_time = time.time() - start_time
    
    X_recon_rand = U_r @ np.diag(S_r) @ Vt_r
    
    error_randomized = np.linalg.norm(X_scaled - X_recon_rand, 'fro')
    
    print(f"Truncated SVD Error  : {error_truncated:.4f}")
    print(f"Randomized SVD Error : {error_randomized:.4f}")
    print(f"Difference in Error  : {abs(error_randomized - error_truncated):.4f}")
    print(f"Randomized SVD Time  : {rand_time:.4f} seconds")
    print(f"Truncated SVD Time   : {trunc_time:.4f} seconds")
    
    return error_randomized

In [21]:
error_randomized = q14_compare_svd(bow_scaled, 42, reconstruction_error, trunc_time)

=== SVD Comparison (k = 42) ===
Truncated SVD Error  : 98.5440
Randomized SVD Error : 115.1949
Difference in Error  : 16.6509
Randomized SVD Time  : 0.0119 seconds
Truncated SVD Time   : 0.1808 seconds


## q15

In [22]:
def q15_top_words_per_component(Vtk, vocabulary_list):
    k = Vtk.shape[0]

    for comp_idx in range(k):
        comp = Vtk[comp_idx]
        top5_idx = np.argsort(np.abs(comp))[::-1][:5]
        top5 = [(vocabulary_list[i], round(comp[i], 3)) for i in top5_idx]
        print(f"Component {comp_idx + 1}: {top5}")

In [23]:
q15_top_words_per_component(Vt_r, vocabulary_list)

Component 1: [('election', np.float64(-0.341)), ('labour', np.float64(-0.327)), ('minister', np.float64(-0.256)), ('party', np.float64(-0.256)), ('technology', np.float64(0.236))]
Component 2: [('digital', np.float64(-0.23)), ('technology', np.float64(-0.227)), ('government', np.float64(-0.225)), ('uk', np.float64(-0.224)), ('market', np.float64(-0.197))]
Component 3: [('time', np.float64(0.295)), ('firm', np.float64(-0.262)), ('win', np.float64(0.26)), ('game', np.float64(0.234)), ('won', np.float64(0.234))]
Component 4: [('film', np.float64(0.339)), ('director', np.float64(0.333)), ('artist', np.float64(0.32)), ('music', np.float64(0.318)), ('won', np.float64(0.308))]
Component 5: [('company', np.float64(-0.291)), ('deal', np.float64(-0.252)), ('artist', np.float64(-0.249)), ('music', np.float64(-0.247)), ('film', np.float64(0.235))]
Component 6: [('growth', np.float64(0.536)), ('economy', np.float64(0.471)), ('expected', np.float64(0.231)), ('game', np.float64(0.204)), ('firm', np.f

## q16

In [24]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def euclidean_dist(a, b):
    return np.linalg.norm(a - b)

In [25]:
def q16_word_pair_similarity(Vtk, vocabulary_list):
    word2idx = {w: i for i, w in enumerate(vocabulary_list)}
    word_vecs = Vtk.T

    pairs = [
        ('mobile', 'technology'), ('director', 'film'), ('win', 'won'),
        ('play', 'game'), ('play', 'law'), ('government', 'music')
    ]

    print(f"{'Pair':<28}{'Cosine Sim':>12}{'Euclidean Dist':>18}")
    for w1, w2 in pairs:
        v1 = word_vecs[word2idx[w1]]
        v2 = word_vecs[word2idx[w2]]
        cs = cosine_sim(v1, v2)
        ed = euclidean_dist(v1, v2)
        print(f"({w1}, {w2}){'':<{max(1,28-len(w1)-len(w2)-4)}}{cs:>12.4f}{ed:>18.4f}")

In [26]:
q16_word_pair_similarity(Vt_r, vocabulary_list)

Pair                          Cosine Sim    Euclidean Dist
(mobile, technology)              0.1647            0.9876
(director, film)                  0.6785            0.5950
(win, won)                        0.0350            1.2931
(play, game)                      0.0216            1.2806
(play, law)                       0.0207            1.3231
(government, music)               0.0193            1.1006


## q17

In [27]:
def q17_document_word_similarity(df_train, bow_train, Uk, Sk, Vtk, doc_idx, fname):
    doc_vec = Uk[doc_idx] * Sk
    word_vecs = Vtk.T

    sims = np.array([cosine_sim(doc_vec, word_vecs[i]) for i in range(len(vocabulary_list))])
    counts = bow_train[doc_idx]
    order = np.argsort(sims)[::-1]

    fig, axes = plt.subplots(2, 1, figsize=(18, 9))
    axes[0].bar(np.array(vocabulary_list)[order], sims[order], color='mediumseagreen')
    axes[0].set_title(f'Cosine Similarity between Document #{doc_idx} and each word (latent space)')
    axes[0].set_ylabel('Cosine Similarity')
    axes[0].tick_params(axis='x', rotation=75)

    axes[1].bar(np.array(vocabulary_list)[order], counts[order], color='indianred')
    axes[1].set_title(f'Word counts in Document #{doc_idx} (same word order)')
    axes[1].set_ylabel('Count')
    axes[1].tick_params(axis='x', rotation=75)

    plt.tight_layout()
    plt.savefig(f'media/{fname}.png', dpi=130)
    plt.close(fig)

In [28]:
q17_document_word_similarity(df_train, bow_train, U_r, S_r, Vt_r, 339, "q17_doc_similarity")

## q19

In [29]:
def q19_category_heatmap(df_train, Uk, Sk, fname):
    doc_vecs = Uk * Sk
    
    categories = sorted(df_train['Label'].unique())

    mean_vecs = []
    for c in categories:
        mask = (df_train['Label'] == c).values
        mean_vecs.append(doc_vecs[mask].mean(axis=0))
        
    mean_vecs = np.array(mean_vecs)
    print(f"Mean latent vectors per category shape: {mean_vecs.shape}")

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(mean_vecs, cmap='coolwarm', center=0,
                yticklabels=[LABELS_MAP[c] for c in categories],
                xticklabels=[f'C{i+1}' for i in range(mean_vecs.shape[1])],
                ax=ax, cbar_kws={'label': 'Mean latent value'})
                
    ax.set_title('Mean Latent Space Vector per Category', fontsize=14)
    ax.set_xlabel('Latent Component', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(f'media/{fname}.png', dpi=130)
    plt.close(fig)
    
    return mean_vecs

In [30]:
q19_category_heatmap(df_train, U_r, S_r, "q19_category_heatmap")

Mean latent vectors per category shape: (5, 42)


array([[-1.99128219e+00, -9.84111645e-01,  8.48473962e-01,
        -2.04164061e-01,  4.63088207e-01, -4.96423864e-01,
        -6.78196486e-02,  1.38042156e-01, -1.84670428e-04,
        -1.94238588e-01,  1.87656225e-02,  2.85158822e-01,
        -7.86233479e-02,  1.14942668e-01, -1.26442541e-01,
         1.07793964e-01, -2.97544708e-02, -1.69650382e-01,
        -3.52883738e-02,  7.11417022e-03, -6.53488799e-02,
        -4.24579779e-02,  8.44691236e-02,  6.62640245e-02,
        -3.93254550e-02, -3.29528368e-02, -1.46090080e-02,
        -4.45163581e-02, -1.23311967e-02, -1.31383091e-02,
         6.63672925e-02,  5.32753686e-02, -2.02835806e-02,
         1.59224484e-01,  1.46887582e-02, -6.03674587e-03,
        -4.77637133e-02,  1.86917630e-02, -5.82791970e-02,
        -1.28301387e-03,  3.40587266e-02, -1.84020741e-02],
       [ 1.20117641e-01,  1.58477372e+00,  5.70380123e-01,
        -4.63547172e-01, -1.71448630e-01,  9.98208019e-02,
         2.02360609e-01,  1.40967406e-01,  4.11284754e-

## q20

In [31]:
def cosine_sim_matrix(A, B):
    A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_norm @ B_norm.T


def q20_evaluate(df_train, bow_train, df_test, bow_test, scaler, Vtk):
    X_train_scaled = scaler.transform(bow_train.astype(float))
    doc_latent_train = X_train_scaled @ Vtk.T

    categories = sorted(df_train['Label'].unique())
    centroids = np.array([
        doc_latent_train[(df_train['Label'] == c).values].mean(axis=0)
        for c in categories
    ])

    X_test_scaled = scaler.transform(bow_test.astype(float))
    doc_latent_test = X_test_scaled @ Vtk.T

    sims = cosine_sim_matrix(doc_latent_test, centroids)
    preds = np.array(categories)[np.argmax(sims, axis=1)]
    true = df_test['Label'].values

    acc_overall = (preds == true).mean()
    print(f"Overall accuracy: {acc_overall:.4f}  (n={len(true)})")
    for c in categories:
        mask = true == c
        if mask.sum() > 0:
            acc_c = (preds[mask] == true[mask]).mean()
            print(f"  {LABELS_MAP[c]:<15} n={mask.sum():<5} acc={acc_c:.4f}")
        else:
            print(f"  {LABELS_MAP[c]:<15} n=0 (not present in this test split)")
    return acc_overall

In [32]:
q20_evaluate(df_train, bow_train, df_test, bow_test, scaler, Vt_r)

Overall accuracy: 0.7600  (n=225)
  Politics        n=50    acc=0.7200
  Sport           n=53    acc=0.9245
  Technology      n=38    acc=0.6842
  Entertainment   n=36    acc=0.6944
  Business        n=48    acc=0.7292


np.float64(0.76)